# Lab 02-01 — Local embeddings: BGE vs E5 on rag-mini-wikipedia

**Track 02 · Embeddings** — the block that decides *which* passages a query can ever reach. An embedding turns text into a vector of numbers, and the whole RAG pipeline stands on the assumption that "similar meaning => similar vector". This lab runs two popular open-source embedders head to head on the same real corpus and inspects the three numbers that decide everything downstream:

* **DIMENSION** — how many floats per vector (768 for both models here). This is the storage cost of your vector database: 768 floats x 4 bytes per chunk.
* **NORM** — the length of the embedding vector. A unit-norm vector makes cosine similarity identical to a dot product, which matters when your vector database offers fast dot-product scoring. BGE is normalized explicitly (`encode_kwargs` `normalize_embeddings=True`); the E5 model on the Hub ships its own `2_Normalize` layer, so it comes out unit-norm too. A surprising number of embedders do NOT normalize — always check.
* **COSINE SIMILARITY** — the retrieval score. We embed a small deterministic subset of passages once, embed 3 real questions from the corpus `test` split, and rank passages by cosine similarity for each model.

This notebook is **self-contained**: it imports LangChain, numpy, pandas, and scikit-learn directly — no repo component library. The two embedders are built right here as small inline wrappers over `HuggingFaceEmbeddings` (BGE with `normalize_embeddings=True`, E5 with its `"query: "` / `"passage: "` instruction prefixes), which is exactly how the shared `src/embeddings/bge.py` and `src/embeddings/e5.py` components work underneath.

Why local models: no API keys, no per-token cost, no data leaving your machine. BGE (`BAAI/bge-base-en-v1.5`) is an English retrieval model trained with normalized embeddings; E5 (`intfloat/multilingual-e5-base`) covers many languages and is trained with instruction prefixes — E5 queries are prefixed `"query: "` and passages `"passage: "` before embedding. Prefix mismatches are a classic silent retrieval killer.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-huggingface`, `numpy`, `pandas`, and `scikit-learn`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   sentence-transformers -> local BGE + E5 embeddings
#   langchain-huggingface -> HuggingFaceEmbeddings (the universal embedder class)
#   pandas                -> reads the passages/test.parquet corpus
#   scikit-learn          -> cosine_similarity for the top-k ranking
%pip install -q sentence-transformers langchain-huggingface pandas scikit-learn


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# LangChain + numpy/pandas/sklearn — the only libraries this notebook needs.
# Nothing is imported from the repo's src/ component library.
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402


class BGEEmbedding(Embeddings):
    """Inline BGE wrapper — mirrors src/embeddings/bge.py.

    bge models require normalized embeddings for cosine similarity; the
    universal HuggingFaceEmbeddings class provides that via encode_kwargs.
    """

    def __init__(self, model_name: str = "BAAI/bge-base-en-v1.5"):
        self.model = HuggingFaceEmbeddings(
            model_name=model_name,
            encode_kwargs={"normalize_embeddings": True},
        )

    def embed_query(self, text: str) -> list[float]:
        return self.model.embed_query(text)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.embed_documents(texts)


class E5Embedding(Embeddings):
    """Inline E5 wrapper — mirrors src/embeddings/e5.py.

    E5 models are trained with instruction prefixes: queries are prefixed
    "query: " and passages "passage: " before embedding.
    """

    def __init__(self, model_name: str = "intfloat/multilingual-e5-base"):
        self.model = HuggingFaceEmbeddings(model_name=model_name)

    def embed_query(self, text: str) -> list[float]:
        return self.model.embed_query(f"query: {text}")

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.embed_documents([f"passage: {text}" for text in texts])


# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `PASSAGES_PATH` and `TEST_PATH` point at the rag-mini-wikipedia parquet files already on disk; `N_PASSAGES = 20` takes a deterministic head of the 3200-passage corpus (keeps runtime low); `QUESTION_IDS = [1606, 1610, 1604]` are real questions from `test.parquet`, picked to match; `TOP_K = 3` is the retrieval depth, `PREVIEW` truncates the passage previews the demo prints next to each hit, and `BGE_MODEL_NAME` / `E5_MODEL_NAME` name the two local models.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the comparison
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
E5_MODEL_NAME = "intfloat/multilingual-e5-base"
N_PASSAGES = 20  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1604]  # real questions from test.parquet, picked to match
TOP_K = 3
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` reads the first `n` passages as `(passage_texts, passage_ids)` — the ids are the parquet row indices, which is what the top-k demo prints next to each hit. `load_questions` returns `(question_id, question_text)` pairs for the requested `test.parquet` rows, in the order given by `QUESTION_IDS`.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


## 3. Embed & compare — helpers shared by both models

`run_model` embeds all passages in one batched call and the questions one at a time (`embed_query`) — because in real RAG each incoming question is embedded individually. `l2_norm` is the Euclidean length of a vector (the number the demo prints as "passage norm" / "query norm"). `top_k_results` ranks passages by cosine similarity to the query and returns the top-k as `(id, score)` pairs, and `preview` flattens a passage onto one line for printing.


In [ ]:
# --------------------------------------------------------------------------
# 3. Embed & compare — helpers shared by both models
# --------------------------------------------------------------------------
def run_model(
    model: object, passages: list[str], questions: list[str]
) -> tuple[list[list[float]], list[list[float]]]:
    """Embed all passages and all questions with one model.

    Returns (passage_vectors, query_vectors). Passages are embedded in one
    batched call; queries one at a time (``embed_query``) because in real RAG
    each incoming question is embedded individually.
    """
    passage_vecs = model.embed_documents(passages)
    query_vecs = [model.embed_query(q) for q in questions]
    return passage_vecs, query_vecs


def l2_norm(vector: list[float]) -> float:
    """Euclidean length of an embedding vector."""
    return float(np.linalg.norm(np.asarray(vector, dtype=np.float32)))


def top_k_results(
    query_vec: list[float],
    passage_vecs: list[list[float]],
    passage_ids: list[int],
    k: int,
) -> list[tuple[int, float]]:
    """Rank passages by cosine similarity to the query; return top-k (id, score)."""
    matrix = np.asarray(passage_vecs, dtype=np.float32)
    sims = cosine_similarity(np.asarray([query_vec], dtype=np.float32), matrix)[0]
    order = np.argsort(sims)[::-1][:k]
    return [(passage_ids[i], float(sims[i])) for i in order]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 4. Run the experiment — embed both models, rank every question

`run_experiment` loads the corpus subset and the three test questions, builds both inline embedders, embeds everything, and computes the top-`TOP_K` retrieval per question per model. Everything the demo and the gate need is returned in one dict — no printing happens here.


In [ ]:
# --------------------------------------------------------------------------
# 4. Run the experiment — embed both models, rank every question
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    models = {"BGE": BGEEmbedding(model_name=BGE_MODEL_NAME),
              "E5": E5Embedding(model_name=E5_MODEL_NAME)}
    question_texts = [qtext for _, qtext in questions]
    embedded = {
        name: run_model(model, passage_texts, question_texts)
        for name, model in models.items()
    }

    # Top-k retrieval per question per model (cosine similarity).
    topk: dict[str, list[list[tuple[int, float]]]] = {}
    for name in models:
        pvecs, qvecs = embedded[name]
        topk[name] = [
            top_k_results(qvecs[i], pvecs, passage_ids, TOP_K)
            for i in range(len(questions))
        ]

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "embedded": embedded,
        "topk": topk,
    }


## 5. Demo — print the artifact

`print_demo(exp)` prints the artifact from four angles: the corpus subset with the three test questions; the embedding vectors — dimension and norm for both models (the numbers that decide storage cost and whether cosine == dot product); the top-`TOP_K` retrieval per question per model with passage previews; and a takeaway on why the two unit-norm, same-dimension models are still not interchangeable.


In [ ]:
# --------------------------------------------------------------------------
# 5. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    passage_texts, passage_ids = exp["passage_texts"], exp["passage_ids"]
    questions = exp["questions"]
    embedded = exp["embedded"]
    topk = exp["topk"]

    print("=" * 66)
    print("Lab 01 — local embeddings: BGE vs E5 on rag-mini-wikipedia")
    print(f"{BGE_MODEL_NAME}  vs  {E5_MODEL_NAME}")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {len(passage_texts)} passages (first {N_PASSAGES} of 3200, "
          f"ids {passage_ids[0]}..{passage_ids[-1]})")
    print(f"    {len(questions)} questions from test.parquet:")
    for qid, qtext in questions:
        print(f"      [{qid}] {qtext}")

    print("\n[2] Embedding vectors — dimension and norm:")
    print(f"    {'model':<6}{'dim':>6}{'passage norm':>14}{'query norm':>14}")
    for name, (pvecs, qvecs) in embedded.items():
        dim = len(pvecs[0])
        p_norm = l2_norm(pvecs[0])
        q_norm = l2_norm(qvecs[0])
        print(f"    {name:<6}{dim:>6}{p_norm:>14.4f}{q_norm:>14.4f}")
    print("    BGE is normalized explicitly (encode_kwargs); E5 unit-norm via its")
    print("    model's own 2_Normalize layer — cosine == dot product for both.")

    print(f"\n[3] Top-{TOP_K} retrieval per question (cosine similarity):")
    for i, (qid, qtext) in enumerate(questions):
        print(f'\n    Q[{qid}] "{qtext}"')
        for name in embedded:
            hits = topk[name][i]
            print(f"      {name:<4} " + "  ".join(
                f"id {pid} {score:.4f}" for pid, score in hits
            ))
            for pid, score in hits:
                idx = passage_ids.index(pid)
                print(f"            {score:.4f}  {preview(passage_texts[idx])}")

    print("\n[4] Takeaway")
    print("    Same dimension (768) and both unit-norm, yet not interchangeable:")
    print("    E5's wrapper adds 'query: '/'passage: ' prefixes that BGE never")
    print("    sees, and BGE's normalization is explicit while E5's is baked")
    print("    into the model. When you swap embedders, check the vector norm")
    print("    and the prefix handling together — and remember retrieval scores")
    print("    are only comparable within one model's vector space, never across")
    print("    models.")


## 6. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `N_PASSAGES` passages and `len(QUESTION_IDS)` questions loaded; both models produce 768-dim, unit-norm vectors (the lab's core claim — cosine == dot product for both); every model-question top-`TOP_K` list has exactly `TOP_K` hits whose ids live in the corpus subset with cosine scores in `[-1, 1]`; and the two models' vectors actually differ. Every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 6. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append((f"exactly {N_PASSAGES} passages loaded (first {N_PASSAGES} of 3200)",
                   len(exp["passage_texts"]) == N_PASSAGES))
    checks.append((f"{len(QUESTION_IDS)} questions loaded from test.parquet",
                   len(exp["questions"]) == len(QUESTION_IDS)))

    for name in ("BGE", "E5"):
        pvecs, qvecs = exp["embedded"][name]
        checks.append((f"{name}: {N_PASSAGES} passage vectors, all 768-dim",
                       len(pvecs) == N_PASSAGES and all(len(v) == 768 for v in pvecs)))
        checks.append((f"{name}: {len(QUESTION_IDS)} query vectors, all 768-dim",
                       len(qvecs) == len(QUESTION_IDS) and all(len(v) == 768 for v in qvecs)))
        checks.append((f"{name}: passage vector is unit-norm (got {l2_norm(pvecs[0]):.4f})",
                       abs(l2_norm(pvecs[0]) - 1.0) < 1e-2))
        checks.append((f"{name}: query vector is unit-norm (got {l2_norm(qvecs[0]):.4f})",
                       abs(l2_norm(qvecs[0]) - 1.0) < 1e-2))
        for i, (qid, _qtext) in enumerate(exp["questions"]):
            hits = exp["topk"][name][i]
            checks.append((f"{name} Q[{qid}]: top-{TOP_K} hits, ids in corpus subset",
                           len(hits) == TOP_K and all(pid in exp["passage_ids"] for pid, _ in hits)))
            checks.append((f"{name} Q[{qid}]: cosine scores in [-1, 1]",
                           all(-1.0 <= s <= 1.0 for _, s in hits)))

    b0 = np.asarray(exp["embedded"]["BGE"][0][0], dtype=np.float32)
    e0 = np.asarray(exp["embedded"]["E5"][0][0], dtype=np.float32)
    checks.append(("BGE and E5 passage vectors differ (two distinct models)",
                   float(np.linalg.norm(b0 - e0)) > 0.1))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A minute or two of local embedding (BGE + E5, both cached on disk) — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The three test questions ranked by both models: dimension, norm, and the top-3 cosine hits with passage previews — the numbers that decide storage cost, scoring metric, and retrieval quality.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
